## Solución analítica para un pulso de inyección en un sistema binario: 
sección 3.3 de [M. De Simoni, J. Carrera and X. Sánchez-Vila, A. Guadagnini, 2005] 
WATER RESOURCES RESEARCH, VOL. 41, W11410, doi:10.1029/2005WR004056, 2005

## Objetivo: Visualizar la solución analítica para un pulso de inyección en un sistema binario, y compararlo con las soluciones numéricas.

Consideramos un medio poroso homogéneo 3D con porosidad constante $n$, un flujo uniforme en la dirección $x$, con velocidad del fluido $ V=\frac{q}{n} $, tensor de dispersión diagonal con dispersividad longitudinal $D_L$ y transversal $D_T$.

Se inyecta de manera puntual un volumen $V_e$, que contiene dos especies disueltas con concentraciones $c_1$ y $c_2$.

Las condiciones de frontera son: lejos del punto de inyección las concentraciones tienden a

$
c_i(x \rightarrow \infty , t)=c_{i0}, \qquad i=1,2
$

La condición inicial se expresa como

$
c_i(x,t=0)=\frac{c_{ie}V_e}{n}\,\delta(x)+c_{i0}
$

El sistema está en equilibrio químico, por lo que

$ c_{10}c_{20}=K,  $ y también $ (c_{10}+c_{1e})(c_{20}+c_{2e})=K $

Se introduce el componente conservativo

$
u=c_1-c_2
$

donde, una vez resuelto $u$, podemos recuperar $c_1$ y $c_2$ como:

$$c_1 = \frac{u+\sqrt{u^2 + 4K}}{2}, \hspace{5mm} c_1 = \frac{-u+\sqrt{u^2 + 4K}}{2}, \tag{especies} $$

Las condiciones para $u$ son

$
u(x \rightarrow \infty ,t)=u_0
$

$
u(x,t=0)=\frac{u_eV_e}{n}\,\delta(x)+u_0
$

La solución, de acuerdo con [Domenico and Schwartz, 1997, p. 380] en [De Simoni, J. Carrera and X. Sánchez-Vila, A. Guadagnini, 2005] es:

$$ u({x}, t) = u_0 + \frac{18}{\sqrt{2\pi}}
\frac{u_e}{\tilde V_d}
\exp\left(-\frac{1}{2}\rho^2\right)  \tag{26} $$

$$\rho = \sqrt{ \frac{(x - V t)^2}{2 t D_L} + \frac{y^2 + z^2}{2 t D_T} }.
\tag{27} $$

$$\tilde{V}_{d} = \frac{72 \pi n \sqrt{2} t^{3/2} \sqrt{D_L} D_T}{V_e}.
\tag{28}$$

Sustituyendo (27) y (28) en (26) y considerando que el problema es 1D (solo X), se obtiene:
$$u({x}, t) = u_0 +
\frac{u_e\,V_e}{8\,n\,\pi^{3/2}\,t^{3/2}\,\sqrt{D_L}\,D_T}
\exp\!\left(
-\frac{(x - V t)^2}{4 t D_L}
\right). \tag{1}
 $$

Que puede ser expresada como:

$$
u(x,t)=u_0+\frac{M(t)}{\sqrt{4\pi D_L t}}\exp\!\left(-\frac{(x-x_0-Vt)^2}{4D_L t}\right), \hspace{10mm} M(t)=\frac{u_e\,V_e}{4n\pi D_T\,t} = M0
$$


## Entonces, la solución analítica (1) puede ser expresada como:

$$u(x,t) = u_0 + \frac{M0}{\sqrt{4\pi D_L t}}
\exp\!\left(
-\frac{(x - x_0 - V t)^2}{4 D_L t}
\right)  \tag{2}
 $$
 $$u(x,t) = u_0 + \frac{M(t)}{\sqrt{4\pi D_L t}}
\exp\!\left(
-\frac{(x - x_0 - V t)^2}{4 D_L t}
\right)  \tag{3}
 $$

In [2]:
# Funcion u(x,t) con M(t)
# =========================================================
# 1. Librerías
# =========================================================
import math
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

# =========================================================
# 2. Malla espacial (centros de celda)
# =========================================================
Nx = 30
x_vals = np.array([i - 0.5 for i in range(1, Nx + 1)], dtype=float)

t_vals = [0, 2, 10, 20, 30, 40]

# =========================================================
# 3. Parámetros físicos
# =========================================================
ue = 1.0
u0 = 1.0
V  = 0.4
Ve = 0.4
DL = 0.4
DT = 0.2
n  = 0.5

x0 = 10.5          # centro del pulso
M0 = 199.0         # "masa" inicial discreta en t=0 (impulso)

# =========================================================
# 4. Función u(x,t)
# =========================================================
def u_sol(x, t):
    # --- Condición inicial discreta en t=0 ---
    if t == 0:
        u = np.full_like(x, u0, dtype=float)
        # poner el impulso exactamente en x=10.5 (si existe) o en la celda más cercana
        idx = int(np.argmin(np.abs(x - x0)))
        u[idx] = u0 + M0
        return u

    # --- Para t>0: tu ecuación (2) ---
    M_t = (ue * Ve ) / (4 * n * math.pi * DT * t)  # M(t)

    #pref = M_t / math.sqrt(4 * math.pi * DL * t)
    pref = M0 / math.sqrt(4 * math.pi * DL * t)
    expo = np.exp(-((x - x0 - V*t)**2) / (4 * DL * t))

    return u0 + pref * expo

# =========================================================
# 5. Precalcular soluciones
# =========================================================
u_pulso = {t: u_sol(x_vals, t) for t in t_vals}

# =========================================================
# 6. Gráfica interactiva
# =========================================================
estado = {t: False for t in t_vals}
out = widgets.Output()

def actualizar():
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7,5))

        for t, on in estado.items():
            if on:
                ax.plot(x_vals, u_pulso[t], marker="o", label=f"t = {t}")

        ax.set_xlabel("x")
        ax.set_ylabel("u(x,t)")
        ax.set_title("u(x,t) con impulso inicial en t=0")
        ax.grid(True)
        if any(estado.values()):
            ax.legend()
        plt.show()

def toggle(change, t):
    estado[t] = change["new"]
    actualizar()

botones = []
for t in t_vals:
    b = widgets.ToggleButton(description=f"t = {t}",
                             layout=widgets.Layout(width="90px"))
    b.observe(lambda c, t=t: toggle(c, t), names="value")
    botones.append(b)


display(widgets.VBox([
    widgets.HTML("<b>Selecciona tiempos:</b>"),
    #widgets.HBox(botones),
    out
]))




## VISUALIZACION DE RESULTADOS ANALITICOS Y NUMERICOS PARA EL YESO

In [3]:
# VISUALIZACION DE RESULTADOS ANALITICOS Y NUMERICOS 
# =========================================================
# LECTURA DE EXCEL + VISUALIZACIÓN MULTI-SOLUCIONES
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# =========================================================
# 1. Leer archivo Excel
# =========================================================
archivo = "solana1.xlsx"

# Leer hoja completa sin encabezados
df = pd.read_excel(archivo, header=None)

# =========================================================
# 2. Coordenada x (E50:E79 → filas 49:79, columna 4)
# =========================================================
x_excel = pd.to_numeric(df.iloc[49:79, 4], errors="coerce").values

# =========================================================
# 3. SAJJ → F50:J79  (t=2,10,20,30,40)
# =========================================================
SAJJ = {
    2:  pd.to_numeric(df.iloc[49:79, 5], errors="coerce").values,
    10: pd.to_numeric(df.iloc[49:79, 6], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79, 7], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79, 8], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79, 9], errors="coerce").values
}

# =========================================================
# 4. SNJJ → K50:N79 (t=10,20,30,40)
# =========================================================
SNJJ = {
    10: pd.to_numeric(df.iloc[49:79,10], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79,11], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79,12], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79,13], errors="coerce").values
}

# =========================================================
# 5. SNBN → O50:R79 (t=10,20,30,40)
# =========================================================
SNBN = {
    10: pd.to_numeric(df.iloc[49:79,14], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79,15], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79,16], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79,17], errors="coerce").values
}

# =========================================================
# 6. Estados independientes para cada grupo
# =========================================================
estado_u    = {t: False for t in u_pulso}
estado_SAJJ = {t: False for t in SAJJ}
estado_SNJJ = {t: False for t in SNJJ}
estado_SNBN = {t: False for t in SNBN}

out = widgets.Output()

# =========================================================
# 7. Función de actualización de gráfica
# =========================================================
def actualizar():
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8,5))

        # u_pulso (tu solución)
        for t, on in estado_u.items():
            if on:
                ax.plot(x_vals, u_pulso[t], "-o", label=f"u_pulso t={t}")

        # SAJJ
        for t, on in estado_SAJJ.items():
            if on:
                ax.plot(x_excel, SAJJ[t], "--s", label=f"SAJJ t={t}")

        # SNJJ
        for t, on in estado_SNJJ.items():
            if on:
                ax.plot(x_excel, SNJJ[t], ":^", label=f"SNJJ t={t}")

        # SNBN
        for t, on in estado_SNBN.items():
            if on:
                ax.plot(x_excel, SNBN[t], "-.d", label=f"SNBN t={t}")

        ax.set_xlabel("x")
        ax.set_ylabel("u(x,t)")
        #ax.set_title("SA-PULSO   vs   SA-JJ")
        #ax.set_title("SA-PULSO   vs   SN-JJ")
        #ax.set_title("SA-PULSO   vs   SN-BN")
        ax.set_title("SN-JJ   vs   SN-BN")
        ax.grid(True)

        if any(estado_u.values()) or any(estado_SAJJ.values()) \
           or any(estado_SNJJ.values()) or any(estado_SNBN.values()):
            ax.legend()

        plt.show()

# =========================================================
# 8. Botones en matriz (tiempos en columnas)
# =========================================================

# todos los tiempos posibles
todos_t = sorted(set(list(u_pulso) + list(SAJJ) + list(SNJJ) + list(SNBN)))

def crear_boton(estado, t):
    if t not in estado:
        return widgets.Label("")  # celda vacía si no existe

    b = widgets.ToggleButton(
        description="",
        value=False,
        layout=widgets.Layout(
            width="40px",
            height="30px",
            border="2px solid black"   # ← marco visible
        ),
        style={"button_color": "white"}  # fondo blanco
    )

    def on_change(change, t=t):
        estado[t] = change["new"]
        actualizar()

    b.observe(on_change, names="value")
    return b


# =========================================================
# 9. Construir tabla de botones
# =========================================================

grid = []

# encabezado
header = [widgets.Label("tipo")] + [widgets.Label(f"t={t}") for t in todos_t]
grid.extend(header)

# filas
filas = [
    ("u_pulso", estado_u),
    ("SAJJ", estado_SAJJ),
    ("SNJJ", estado_SNJJ),
    ("SNBN", estado_SNBN),
]

for nombre, estado in filas:
    grid.append(widgets.Label(nombre))
    for t in todos_t:
        grid.append(crear_boton(estado, t))


# layout tipo tabla
tabla_botones = widgets.GridBox(
    grid,
    layout=widgets.Layout(
        grid_template_columns=f"120px repeat({len(todos_t)}, 60px)",
        grid_gap="5px 5px"
    )
)

display(widgets.VBox([tabla_botones, out]))


## VISUALIZACIÓN DE CONCENTRACIONES $c_1$ Y $c_2$ PARA ESPECIES DEL YESO 
## COMPARACIÓN DE CONCENTRACIONES UTILIZANDO u_pulso y SNBN

Para obtener las especies $c_1$ y $c_2$ utilizamos las expresiones:

$c_1 = \frac{u+\sqrt{u^2 + 4K}}{2},$ $\hspace{5mm}$  $c_1 = \frac{-u+\sqrt{u^2 + 4K}}{2}$

Para propósitos de comparación, utilizaremos la $u$ mediante u_pulso y la $u$ obtenida mediante SNBN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


class VisualizadorEspecies:

    def __init__(
        self,
        x_vals,
        x_excel,
        t_vals,
        u_pulso,
        SNBN
    ):

        self.x_vals = x_vals
        self.x_excel = x_excel

        self.t_vals = list(t_vals)

        self.u_pulso = u_pulso
        self.SNBN = SNBN

        self.K = 1 / 30359.9

        self.indice = 0

        self.modo = "ambas"

        self.out = widgets.Output()

        self.titulo = widgets.HTML()

        # ----------------

        self.btn_prev = widgets.Button(
            description="◀ Tiempo"
        )

        self.btn_next = widgets.Button(
            description="Tiempo ▶"
        )

        self.btn_c1 = widgets.Button(
            description="c₁"
        )

        self.btn_c2 = widgets.Button(
            description="c₂"
        )

        self.btn_ambas = widgets.Button(
            description="c₁ + c₂"
        )

        # ----------------

        self.btn_prev.on_click(
            self.anterior
        )

        self.btn_next.on_click(
            self.siguiente
        )

        self.btn_c1.on_click(
            self.ver_c1
        )

        self.btn_c2.on_click(
            self.ver_c2
        )

        self.btn_ambas.on_click(
            self.ver_ambas
        )

        self.ui = widgets.VBox([

            widgets.HTML(
            """
            <h2 style='text-align:center'>
            Visualización de especies c1, c2, usando u_pulso y SNBN
            </h2>
            """
            ),

            self.titulo,

            self.out,

            widgets.VBox([

                widgets.HBox([
                    self.btn_prev,
                    self.btn_next
                ]),

                widgets.HBox([
                    self.btn_c1,
                    self.btn_c2,
                    self.btn_ambas
                ])

            ])

        ])

        self.actualizar()

    # =================================

    def especies(self, u):

        c1 = (
            u
            + np.sqrt(
                u**2
                + 4*self.K
            )
        ) / 2

        c2 = (
            -u
            + np.sqrt(
                u**2
                + 4*self.K
            )
        ) / 2

        return c1, c2

    # =================================

    def actualizar(self):

        with self.out:

            self.out.clear_output(
                wait=True
            )

            t = self.t_vals[
                self.indice
            ]

            fig, ax = plt.subplots(
                figsize=(10,6)
            )

            # -------------------

            u = np.asarray(
                self.u_pulso[t]
            )

            c1_u, c2_u = (
                self.especies(u)
            )

            existe = (
                t in self.SNBN
            )

            if existe:

                sn = np.asarray(
                    self.SNBN[t]
                )

                c1_sn, c2_sn = (
                    self.especies(sn)
                )

            # ===================
            # c1
            # ===================

            if self.modo in [
                "c1",
                "ambas"
            ]:

                ax.plot(

                    self.x_vals,

                    c1_u,

                    "-o",

                    linewidth=2,

                    markersize=6,

                    label=f"c₁ ← u_pulso"

                )

                if existe:

                    ax.plot(

                        self.x_excel,

                        c1_sn,

                        "--o",

                        linewidth=2,

                        markersize=6,

                        label=f"c₁ ← SNBN"

                    )

            # ===================
            # c2
            # ===================

            if self.modo in [
                "c2",
                "ambas"
            ]:

                ax.plot(

                    self.x_vals,

                    c2_u,

                    "-s",

                    linewidth=2,

                    markersize=6,

                    label=f"c₂ ← u_pulso"

                )

                if existe:

                    ax.plot(

                        self.x_excel,

                        c2_sn,

                        "--s",

                        linewidth=2,

                        markersize=6,

                        label=f"c₂ ← SNBN"

                    )

            # -------------------

            self.titulo.value = (
                f"""
                <h3 style='text-align:center'>
                Tiempo = {t}
                |
                {self.modo}
                </h3>
                """
            )

            ax.set_xlabel(
                "x"
            )

            ax.set_ylabel(
                "Concentración"
            )

            ax.grid()

            ax.legend()

            plt.show()

    # =================================

    def anterior(
        self,
        b
    ):

        if (
            self.indice > 0
        ):

            self.indice -= 1

        self.actualizar()

    def siguiente(
        self,
        b
    ):

        if (
            self.indice
            <
            len(
                self.t_vals
            )-1
        ):

            self.indice += 1

        self.actualizar()

    def ver_c1(
        self,
        b
    ):

        self.modo = "c1"

        self.actualizar()

    def ver_c2(
        self,
        b
    ):

        self.modo = "c2"

        self.actualizar()

    def ver_ambas(
        self,
        b
    ):

        self.modo = "ambas"

        self.actualizar()


vista_especies = (

    VisualizadorEspecies(

        x_vals,

        x_excel,

        t_vals,

        u_pulso,

        SNBN

    )

)

display(
    vista_especies.ui
)

## VISUALIZACIÓN DE CONCENTRACIONES $c_1$ Y $c_2$ PARA ESPECIES DEL YESO 
## COMPARACIÓN DE CONCENTRACIONES UTILIZANDO u_pulso, SNBN y SNJJ

De manera análoga al caso anterior, para obtener las especies $c_1$ y $c_2$ utilizamos las expresiones:

$c_1 = \frac{u+\sqrt{u^2 + 4K}}{2},$ $\hspace{5mm}$  $c_1 = \frac{-u+\sqrt{u^2 + 4K}}{2}.$

Al igual que en el caso anterior, utilizaremos la $u$ mediante u_pulso, la $u$ obtenida mediante SNBN y la $u$ obtenida mediante SNJJ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


class VisualizadorEspecies3:

    def __init__(
        self,
        x_vals,
        x_excel,
        t_vals,
        u_pulso,
        SNBN,
        SNJJ
    ):

        self.x_vals = x_vals
        self.x_excel = x_excel

        self.t_vals = sorted(

            set(u_pulso.keys())
            |
            set(SNBN.keys())
            |
            set(SNJJ.keys())

        )

        self.u_pulso = u_pulso

        self.SNBN = SNBN

        self.SNJJ = SNJJ

        self.K = 1/30359.9

        self.indice = 0

        self.modo = "ambas"

        self.out = widgets.Output()

        self.titulo = widgets.HTML()

        # ------------------

        self.btn_prev = widgets.Button(
            description="◀ Tiempo"
        )

        self.btn_next = widgets.Button(
            description="Tiempo ▶"
        )

        self.btn_c1 = widgets.Button(
            description="c₁"
        )

        self.btn_c2 = widgets.Button(
            description="c₂"
        )

        self.btn_ambas = widgets.Button(
            description="c₁+c₂"
        )

        self.btn_prev.on_click(
            self.anterior
        )

        self.btn_next.on_click(
            self.siguiente
        )

        self.btn_c1.on_click(
            self.ver_c1
        )

        self.btn_c2.on_click(
            self.ver_c2
        )

        self.btn_ambas.on_click(
            self.ver_ambas
        )

        self.ui = widgets.VBox([

            widgets.HTML(
            """
            <h2 style='text-align:center'>
            Visualización de especies c1 y c2 
            usando u_pulso, SNBN y SNJJ
            </h2>
            """
            ),

            self.titulo,

            self.out,

            widgets.HBox([

                self.btn_prev,

                self.btn_next

            ]),

            widgets.HBox([

                self.btn_c1,

                self.btn_c2,

                self.btn_ambas

            ])

        ])

        self.actualizar()

    # =====================

    def especies(self,u):

        c1 = (
            u
            +
            np.sqrt(
                u**2
                +
                4*self.K
            )
        )/2

        c2 = (
            -u
            +
            np.sqrt(
                u**2
                +
                4*self.K
            )
        )/2

        return c1,c2

    # =====================

    def dibujar(
        self,
        ax,
        x,
        datos,
        nombre,
        estilo1,
        estilo2
    ):

        c1,c2 = (
            self.especies(
                np.asarray(
                    datos
                )
            )
        )

        if self.modo in [
            "c1",
            "ambas"
        ]:

            ax.plot(

                x,

                c1,

                estilo1,

                linewidth=2,

                markersize=6,

                label=f"c₁ ← {nombre}"

            )

        if self.modo in [
            "c2",
            "ambas"
        ]:

            ax.plot(

                x,

                c2,

                estilo2,

                linewidth=2,

                markersize=6,

                label=f"c₂ ← {nombre}"

            )

    # =====================

    def actualizar(self):

        with self.out:

            self.out.clear_output(
                wait=True
            )

            t = self.t_vals[
                self.indice
            ]

            fig,ax = plt.subplots(
                figsize=(10,6)
            )

            if t in self.u_pulso:

                self.dibujar(

                    ax,

                    self.x_vals,

                    self.u_pulso[t],

                    "u_pulso",

                    "-o",

                    "-s"

                )

            if t in self.SNBN:

                self.dibujar(

                    ax,

                    self.x_excel,

                    self.SNBN[t],

                    "SNBN",

                    "--o",

                    "--s"

                )

            if t in self.SNJJ:

                self.dibujar(

                    ax,

                    self.x_excel,

                    self.SNJJ[t],

                    "SNJJ",

                    ":o",

                    ":s"

                )

            self.titulo.value = (
                f"""
                <h3>
                t={t}
                |
                {self.modo}
                </h3>
                """
            )

            ax.set_xlabel(
                "x"
            )

            ax.set_ylabel(
                "Concentración"
            )

            ax.grid()

            ax.legend()

            plt.show()

    # =====================

    def anterior(
        self,
        b
    ):

        if self.indice>0:

            self.indice-=1

        self.actualizar()

    def siguiente(
        self,
        b
    ):

        if (
            self.indice
            <
            len(
                self.t_vals
            )-1
        ):

            self.indice+=1

        self.actualizar()

    def ver_c1(
        self,
        b
    ):

        self.modo="c1"

        self.actualizar()

    def ver_c2(
        self,
        b
    ):

        self.modo="c2"

        self.actualizar()

    def ver_ambas(
        self,
        b
    ):

        self.modo="ambas"

        self.actualizar()


vista_especies_3 = (

    VisualizadorEspecies3(

        x_vals,

        x_excel,

        t_vals,

        u_pulso,

        SNBN,

        SNJJ

    )

)

display(
    vista_especies_3.ui
)

## Valores numéricos de: u_pulso, SNBN, SNJJ y c1 y c2 para cada t_vals

In [ ]:
# =========================================================
# Guardar u_pulso, SNBN, SNJJ y especies c1,c2
# =========================================================

K = 1/30359.9

with open("soluciones_c1_c2.dat", "w") as f:

    f.write("====================================================\n")
    f.write("u_pulso, SNBN, SNJJ, c1 y c2\n")
    f.write("====================================================\n\n")

    f.write("Para t=0 y t=2, c1 y c2 solo se calculan con u_pulso.\n")
    f.write("Para t=10,20,30,40, c1 y c2 se calculan con u_pulso, SNBN y SNJJ.\n\n")

    # =====================================================
    # Recorrer tiempos
    # =====================================================

    for t in t_vals:

        f.write("####################################################\n")
        f.write(f"# t = {t}\n")
        f.write("####################################################\n\n")

        # =================================================
        # u_pulso
        # =================================================

        if t in u_pulso:

            f.write("----- u_pulso -----\n")
            f.write("       x            u            c1            c2\n")

            u_vals = np.asarray(u_pulso[t])

            c1 = (
                u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            c2 = (
                -u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            for x,u,v1,v2 in zip(
                x_vals,
                u_vals,
                c1,
                c2
            ):

                f.write(
                    f"{x:12.6f} "
                    f"{u:14.6f} "
                    f"{v1:14.6f} "
                    f"{v2:14.6f}\n"
                )

            f.write("\n")

        # =================================================
        # SNBN
        # =================================================

        if t in SNBN:

            f.write("----- SNBN -----\n")
            f.write("       x            u            c1            c2\n")

            u_vals = np.asarray(SNBN[t])

            c1 = (
                u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            c2 = (
                -u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            for x,u,v1,v2 in zip(
                x_excel,
                u_vals,
                c1,
                c2
            ):

                f.write(
                    f"{x:12.6f} "
                    f"{u:14.6f} "
                    f"{v1:14.6f} "
                    f"{v2:14.6f}\n"
                )

            f.write("\n")

        # =================================================
        # SNJJ
        # =================================================

        if t in SNJJ:

            f.write("----- SNJJ -----\n")
            f.write("       x            u            c1            c2\n")

            u_vals = np.asarray(SNJJ[t])

            c1 = (
                u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            c2 = (
                -u_vals
                +
                np.sqrt(u_vals**2 + 4*K)
            )/2

            for x,u,v1,v2 in zip(
                x_excel,
                u_vals,
                c1,
                c2
            ):

                f.write(
                    f"{x:12.6f} "
                    f"{u:14.6f} "
                    f"{v1:14.6f} "
                    f"{v2:14.6f}\n"
                )

            f.write("\n")

print("Archivo guardado: soluciones_c1_c2.dat")